# RAG Practice on PDF Documents

This is the practice code for the blog post [So, What is RAG](https://seojuny.dev/en/what-is-rag). Given the employment-rules PDF (`../../data/en/employment-rules.pdf`) of a fictional company, **Gureumtech**, we build an internal Q&A that answers employees' questions so they don't have to dig through the rulebook themselves.

With no framework (LangChain, etc.) — just `pypdf` + `openai` + `chromadb` — you watch text turn into vectors and get retrieved firsthand.

The flow has two stages.

- **Indexing** (1–3): store the document in the vector DB before any question arrives
- **Answering** (4–5): retrieve and answer each time a question comes in

## 0. Environment Setup (do this once in the terminal first)

To run the notebook cells, **a kernel must exist first.** Run the steps below **in a terminal** once to create the environment and kernel, then pick that kernel in this notebook.

### 1) Check your Python version
**Python 3.10 or higher** is fine. (Verified working on 3.14 as well.)

```bash
python --version        # confirm it is 3.10 or higher
```

If it is too low (3.9 or below), bump it with `pyenv`. (If you don't have `pyenv`, run `brew install pyenv` first.)

```bash
pyenv install 3.12.7    # example. any version 3.10+ is OK
```

### 2) Install pipenv + the libraries in one go
If you don't have `pipenv`, install it first. `pipenv install` creates the virtual environment and installs the packages (from `Pipfile`) in one step.

```bash
pip install --user pipenv      # only if you don't have it. (or: brew install pipenv)

cd rag-pdf-practice
pipenv install                 # installs the exact versions from Pipfile.lock, so everyone gets the same environment
```

### 3) Register the Jupyter kernel
Make **RAG PDF Practice** show up in the kernel list of VS Code / Cursor.

```bash
pipenv run python -m ipykernel install --user --name rag-pdf-practice --display-name "RAG PDF Practice"
```

### 4) Choose a model — OpenAI or local Ollama

**(A) To use OpenAI**, add your API key.

```bash
cp .env.example .env           # open .env and fill in OPENAI_API_KEY (sk-... )
```

**(B) To use a local model (free)**, leave `OPENAI_API_KEY` in `.env` **empty** and install [Ollama](https://ollama.com), then pull the models. With no key, `get_client()` connects to Ollama automatically.

```bash
ollama pull qwen2.5            # answer-generation model
ollama pull nomic-embed-text  # embedding model
# ollama must be running in the background (http://localhost:11434)
```

Once setup is done, select the **RAG PDF Practice** kernel (top-right in VS Code / top bar in Jupyter) and run the cells below in order.

In [ ]:
# Setup — create the LLM client (auto-selects OpenAI or local Ollama)
#   the 'RAG PDF Practice' kernel must be selected (see step 0, Environment Setup)
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # read OPENAI_API_KEY from .env into the environment

def get_client():
    """Use OpenAI if OPENAI_API_KEY is set, otherwise local Ollama.
    Returns: (client, chat model, embedding model) — both embeddings and answers go through the same client."""
    if os.getenv("OPENAI_API_KEY", "").strip().startswith("sk-"):
        client = OpenAI()
        chat_model = "gpt-4o-mini"
        embedding_model = "text-embedding-3-small"
        print("✅ Connected to OpenAI — answer:", chat_model, "| embedding:", embedding_model)
    else:
        # Ollama's OpenAI-compatible endpoint. Pull the models first:
        #   ollama pull qwen2.5 && ollama pull nomic-embed-text
        # (embeddings are sent as a list in one call, so a recent Ollama that supports batch input is recommended)
        client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        chat_model = "qwen2.5"
        embedding_model = "nomic-embed-text"
        print("✅ Connected to local Ollama — answer:", chat_model, "| embedding:", embedding_model)
    return client, chat_model, embedding_model

client, CHAT_MODEL, EMBEDDING_MODEL = get_client()

## ⚙️ CONFIG — experiment by changing only the values here

Change the values in this cell, re-run the cells below, and watch how the answer changes.

| Value | Meaning |
|---|---|
| `CHUNK_SIZE` | Number of characters per chunk. Too large and one chunk mixes several topics so retrieval blurs; too small and context gets cut off |
| `OVERLAP` | Characters shared between adjacent chunks. Prevents sentences from being cut at the boundary |
| `N_RESULTS` | Number of chunks to retrieve (top-k). Too few and the evidence is missing; too many and noise creeps in |
| `SYSTEM_PROMPT` | The answering rules given to the model. Tune hallucination suppression, forced citation, etc. here |
| `TEMPERATURE` | Closer to 0 sticks to the document; higher is more free-form. For factual queries, 0–0.3 is recommended |

Changing `CHUNK_SIZE`·`OVERLAP` changes the vector DB itself, so you must re-run the **2 (chunking) → 3 (indexing)** cells. `N_RESULTS`·`SYSTEM_PROMPT`·`TEMPERATURE` are used only at query time, so re-running just the **question cell** is enough.

In [ ]:
# ── Document ──────────────────────────────
PDF_PATH = "../../data/en/employment-rules.pdf"

# The models (answer/embedding) are chosen automatically by get_client() in the 'Setup' cell above,
# based on the backend (OpenAI/Ollama). To use different models, edit the model names inside get_client().

# ── Chunking (re-indexing needed if changed) ──
CHUNK_SIZE = 300   # the sample doc is small; ~300 chars keeps retrieval roughly at the article level. Try raising it to see retrieval blur
OVERLAP = 60

# ── Retrieval/generation (no re-indexing needed if changed) ──
N_RESULTS = 3
TEMPERATURE = 0
SYSTEM_PROMPT = "Answer the question based on the document below. If it is not in the document, say you don't know."

## 1. Extract text from the document

Pull the per-page text out of the PDF and join it into one string. PDFs with many tables or images can extract poorly, so it is worth eyeballing this result once.

In [ ]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
text = "\n".join(page.extract_text() or "" for page in reader.pages)

print(f"Extracted {len(text)} characters\n")
print(text[:300], "...")

## 2. Chunk the text

Embedding the whole document lumps many topics into one vector and blurs retrieval, so we split it into small pieces. But **smaller is not always better** — `CHUNK_SIZE` causes problems if it leans too far either way.

- **Too large**: one piece mixes several topics, so a chunk gets dragged in whole even when only part of it is relevant, and retrieval blurs.
- **Too small**: sentences and context get cut, so the very information the answer needs is severed at a chunk boundary.

The key is finding a size that matches how much information a single question needs. `OVERLAP` patches the context cut at a boundary by overlapping a little into the next piece. Here we split simply by character count. (Re-run this cell if you changed `CHUNK_SIZE` or `OVERLAP`.)

In [ ]:
def chunk_text(text, chunk_size, overlap):
    assert overlap < chunk_size, "OVERLAP must be smaller than CHUNK_SIZE (otherwise it loops forever)"
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

print(f"{len(chunks)} chunks (chunk_size={CHUNK_SIZE}, overlap={OVERLAP})\n")

# Inspect every chunk. If a chunk is long, show only the start and mark it with ...
PREVIEW = 120
for i, c in enumerate(chunks):
    head = c[:PREVIEW].replace("\n", " ")
    more = " ..." if len(c) > PREVIEW else ""
    print(f"[{i}] ({len(c)} chars) {head}{more}")

## 3. Embed and store in the vector DB

Feed each chunk to the embedding model to turn it into a meaning-carrying numeric vector, and store it — together with the original text — in the vector DB (Chroma). Since we need to pull the original text of a retrieved vector and hand it to the model, we also put the original text into `documents`.

To avoid leftover chunks when you change the chunking settings and re-run, we recreate the collection from scratch every time.

In [ ]:
import chromadb

def build_index(chunks):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks)
    embeddings = [d.embedding for d in resp.data]

    db = chromadb.PersistentClient(path="./chroma_db")
    try:
        db.delete_collection("docs")  # clear previous contents when re-indexing
    except Exception:
        pass
    # hnsw:space="cosine" → measure retrieval distance as cosine distance (similarity = 1 - distance)
    collection = db.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=[f"chunk-{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks,
    )
    return collection

collection = build_index(chunks)
print(f"Indexing done — {collection.count()} chunks stored")

## 4–5. Embed the question · similarity search · generate the answer

The question is turned into a vector with the **same embedding model**, the nearest `N_RESULTS` chunks are found, and the model generates an answer using those chunks as context.

- The answering rules (instructions) go in `system`; the retrieved context and the question go in `user`.
- The one line in `SYSTEM_PROMPT` — "if it is not in the document, say you don't know" — suppresses hallucination.
- Calling with `show_context=True` also shows the retrieved chunks and their **cosine similarity scores** (closer to 1 = more similar to the question). You can see with your own eyes what evidence the answer used and whether retrieval worked.

`ask()` reads `N_RESULTS`·`SYSTEM_PROMPT`·`TEMPERATURE` from `CONFIG` on every call, so if you changed only those, you can re-run this cell and the question cell below without re-indexing.

In [ ]:
def ask(question, show_context=False):
    q_emb = client.embeddings.create(
        model=EMBEDDING_MODEL, input=question
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[q_emb],
        n_results=N_RESULTS,
        include=["documents", "distances"],  # also fetch the distances
    )
    retrieved = results["documents"][0]
    distances = results["distances"][0]
    similarities = [1 - d for d in distances]  # cosine space: similarity = 1 - distance (closer to 1 = more similar)

    if show_context:
        print(f"=== {len(retrieved)} retrieved chunks (most similar first) ===")
        for i, (c, s) in enumerate(zip(retrieved, similarities)):
            print(f"[{i}] similarity {s:.3f} | {c[:90].strip()} ...")
        print("=" * 40, "\n")

    context = "\n\n".join(retrieved)
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"[Document]\n{context}\n\n[Question]\n{question}"},
        ],
        temperature=TEMPERATURE,
    )
    return resp.choices[0].message.content

In [ ]:
question = "How many days of annual leave can I use?"
print(ask(question, show_context=True))

## With RAG vs. without RAG

If you ask the same question without RAG, the model — not knowing the PDF — answers in generalities or makes up something false. Let's compare the two side by side.

In [ ]:
def ask_without_rag(question):
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=TEMPERATURE,
    )
    return resp.choices[0].message.content

print("🚫 Without RAG:\n", ask_without_rag(question), "\n")
print("✅ With RAG:\n", ask(question))

## 🧪 Experiment for yourself

Change the values in the `CONFIG` cell above and watch how the answer changes.

**Changes that need no re-indexing** (`N_RESULTS`, `SYSTEM_PROMPT`, `TEMPERATURE`)
→ run the `CONFIG` cell → re-run only the `ask(...)` cell

**Changes that need re-indexing** (`CHUNK_SIZE`, `OVERLAP`)
→ run the `CONFIG` cell → re-run the **2 (chunking) → 3 (indexing)** cells → `ask(...)`

The cell below is for firing several questions at once. A few things to try:

- Dropping `N_RESULTS` to 1 often loses the evidence and yields "I don't know." What about raising it to 5–8?
- Add `"At the end of the answer, cite the article number that supports it, in the form (Article N)."` to `SYSTEM_PROMPT` to force citations.
- Drop `CHUNK_SIZE` to 100 so articles get cut — how does retrieval waver?
- Ask something not in the document ("What floor is the company parking lot on?") to see whether hallucination is suppressed.

In [ ]:
questions = [
    "How many days off do I get when my spouse gives birth?",
    "How many days a week can I work remotely?",
    "When do I get paid?",
    "What floor is the company parking lot on?",  # not in the document — should answer that it doesn't know
]

for q in questions:
    print(f"Q. {q}")
    print(f"A. {ask(q)}\n")